# 12 · Silver: checks, quarantine, de-duplication and history

**Who this is for:** you, after notebook 11.
**Runs on:** Databricks serverless (costs credits: keep it short, detach when done). Run the pipeline `heavden-medallion` first; every cell here reads tables it wrote.

**By the end you will be able to:**
- write **expectations** and read their results from the pipeline's event log
- explain the **quarantine** pattern, and why a rule must never evaluate to `NULL`
- explain what the de-duplication **watermark** costs, and how the reconciliation table catches it
- link readings to patients when **devices are reused**
- query **SCD type 2** history (`__START_AT` / `__END_AT`) and say what it can't know

The transformations are in `databricks/src/heavden/pipelines/silver.py` (tested on local Spark against the pandas reference in `tests/test_silver.py`); the pipeline files in `databricks/src/pipelines/medallion/` only add the Lakeflow decorators.

## 1. Setup

In [ ]:
import sys

# Find the repo folder this notebook was deployed into, and import the real code from it.
notebook_path = (
    dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
)
repo_root = "/Workspace" + notebook_path.split("/notebooks/databricks/")[0]
sys.path.insert(0, f"{repo_root}/databricks/src")

from heavden.pipelines import silver

dbutils.widgets.text("catalog", "heavden_dev")
CATALOG = dbutils.widgets.get("catalog")
LANDING = f"/Volumes/{CATALOG}/bronze/landing"
print(repo_root, CATALOG, LANDING)

## 2. Expectations: the rules a reading must pass

Each rule is a SQL condition. In the pipeline, `@dp.expect_all_or_drop(READING_RULES)` drops rows that fail and **counts** them. A missing vital is allowed (`heart_rate IS NULL OR ...`): the features handle gaps, and dropping a whole reading because one sensor was off would lose the others.

In [ ]:
for name, condition in silver.READING_RULES.items():
    print(f"{name:22} {condition}")

Why "never `NULL`"? `heart_rate BETWEEN 20 AND 250` is `NULL` when `heart_rate` is missing, and SQL treats `NULL` as "not true". Expectations and the quarantine filter could then disagree about the same row. Every rule here returns `TRUE` or `FALSE` (a test checks it).

## 3. Expectation results from the event log

The pipeline writes its event log to `monitoring.medallion_event_log`. Expectation counts sit in `flow_progress` events; this is what the dashboard's data-quality panel will read.

In [ ]:
display(
    spark.sql(f"""
    WITH progress AS (
        SELECT from_json(
                   details:flow_progress.data_quality.expectations,
                   'array<struct<name string, dataset string,
                                 passed_records bigint, failed_records bigint>>'
               ) AS checks
        FROM {CATALOG}.monitoring.medallion_event_log
        WHERE event_type = 'flow_progress'
          AND details:flow_progress.data_quality.expectations IS NOT NULL
    )
    SELECT c.dataset, c.name, sum(c.passed_records) AS passed, sum(c.failed_records) AS failed
    FROM progress LATERAL VIEW explode(checks) AS c
    GROUP BY 1, 2 ORDER BY 1, 2
""")
)

## 4. Quarantine: keep the bad rows, with reasons

Dropping is right for Silver, but a dropped row can't be investigated. `silver.vitals_quarantine` reads the same Bronze stream and keeps the rows that fail, with `failed_rules`. Simulated data is clean, so it's empty until something goes wrong; the cell below shows what a row would look like, without writing anything.

In [ ]:
example = spark.createDataFrame(
    [("DEV-X-1", "2026-11-01 08:00:00", 400.0, 101.0)],
    "device_id string, ts string, heart_rate double, spo2 double",
)
display(
    silver.quarantine(silver.typed_readings(example), silver.READING_RULES).select(
        "device_id", "heart_rate", "spo2", "failed_rules"
    )
)
display(spark.table(f"{CATALOG}.silver.vitals_quarantine").groupBy("failed_rules").count())

## 5. De-duplication, the watermark, and the reconciliation

A gateway can resend a reading, so `silver.device_readings` keeps one per (`device_id`, `ts`). In a stream, Spark must **remember** the keys it has seen. The **watermark** (24 hours behind the newest `ts`) lets it forget older keys, so memory stays small.

The price: a reading more than 24 hours behind the newest one is **dropped by Spark without a trace**. That only happens if older data is uploaded after newer data (e.g. backfilling a past day). So `silver.vitals_reconciliation` accounts for every Bronze reading, and its expectation `no_missing_readings` flags any loss.

In [ ]:
display(spark.table(f"{CATALOG}.silver.vitals_reconciliation"))

Read it left to right: `bronze_rows` = `quarantined` + `duplicates` + `in_silver` + `missing_from_silver`. Then `in_silver` = `linked` + `not_on_a_patient`. In dev you should see about 15 readings `not_on_a_patient`: taken at the exact minute a stay ended, and dropped by design (`start_ts <= ts < end_ts`, the same rule as `features.py`).

**If `missing_from_silver` > 0:** run the pipeline with a full refresh. `missing_first_ts` / `missing_last_ts` show which period was lost.

## 6. Linking readings to patients

Devices are reused: the same monitor goes to the next patient after a discharge. Joining on `device_id` alone would mix patients up. A reading belongs to the assignment where `start_ts <= ts < end_ts` on its device.

In [ ]:
display(
    spark.sql(f"""
    SELECT device_id, count(*) AS stays, collect_list(encounter_id) AS encounters
    FROM {CATALOG}.silver.device_assignments WHERE __END_AT IS NULL
    GROUP BY device_id HAVING count(*) > 1 ORDER BY stays DESC LIMIT 5
""")
)

`silver.vitals` is a **materialized view**, not a stream. If a vitals file arrives before `sql_ingest` has copied the device assignment, a streaming join would drop those readings for good; a materialized view links them on the next refresh. The cost is recomputing the join, which is small here (and Databricks refreshes incrementally where it can).

## 7. History with AUTO CDC (SCD type 2)

`silver.encounters` keeps every version of each stay. `AUTO CDC` orders the Bronze rows by `(last_updated, _change_version, _ingest_ts)` and sets `__START_AT` / `__END_AT`; the current version has `__END_AT IS NULL`. Re-reading the same row (a snapshot after a reload) doesn't create a new version, because the ingestion bookkeeping columns are excluded from history tracking.

In [ ]:
display(
    spark.sql(f"""
    SELECT encounter_id, unit_id, status, discharge_ts,
           __START_AT.last_updated AS valid_from, __END_AT.last_updated AS valid_to
    FROM {CATALOG}.silver.encounters
    WHERE encounter_id IN (
        SELECT encounter_id FROM {CATALOG}.silver.encounters GROUP BY 1 HAVING count(*) > 1
    )
    ORDER BY encounter_id, valid_from LIMIT 20
""")
)  # empty after a single load: see the next cell

**What history can't know.** Azure SQL keeps only each stay's *current* unit. When two simulated days are loaded at once, a transfer that happened inside those two days is invisible: Silver sees one version, with the later unit. History builds up only when ingestion runs between changes, as it does in drift episodes. A real hospital solves this with an admissions/transfers event feed.

**Snapshots are upserts.** AUTO CDC can't see a row that a snapshot *no longer contains*. The source never deletes, so this only matters after reloading *different* data, which needs a full refresh anyway.

## 8. Try it yourself

**a)** Write a point-in-time query: which unit was encounter `E-100001` on at `2026-11-01 12:00`? Use `__START_AT` / `__END_AT` (remember `__END_AT` is `NULL` for the current version).

In [ ]:
# Try it yourself
unit_at_sql = None

In [ ]:
# Answer (run after trying)
unit_at_sql = f"""
    SELECT encounter_id, unit_id
    FROM {CATALOG}.silver.encounters
    WHERE encounter_id = 'E-100001'
      AND __START_AT.last_updated <= TIMESTAMP'2026-11-01 12:00:00'
      AND (__END_AT IS NULL OR __END_AT.last_updated > TIMESTAMP'2026-11-01 12:00:00')
"""
display(
    spark.sql(unit_at_sql)
)  # no row if the stay's only version starts after 12:00 (its admission, transfer or discharge)

**b)** Add a rule to `READING_RULES` that a reading's `ts` is not in the future, as a SQL condition that is never `NULL`. Check it on `example` above.

In [ ]:
# Try it yourself
rule = None

In [ ]:
# Answer (run after trying)
rule = {"ts_not_in_future": "ts IS NULL OR ts <= current_timestamp() + INTERVAL 5 MINUTES"}
display(
    silver.quarantine(silver.typed_readings(example), silver.READING_RULES | rule).select(
        "failed_rules"
    )
)
# Careful: simulated time is November 2026. Before then, every reading is "in the future".

**c)** Without running it: what would happen to `silver.vitals_reconciliation` if you uploaded `landing/dev/vitals` files for a day **three days before** the newest data, and ran the pipeline (not a full refresh)?

## 9. Pitfalls

**Pitfall: rules that return `NULL`.** `NOT (x BETWEEN 20 AND 250)` is `NULL` when `x` is missing, so the row escapes both the drop and the quarantine. Write `x IS NULL OR x BETWEEN ...`.

**Pitfall: stream-static joins for late-arriving dimensions.** A stream joined to a table matches only what the table holds at that moment. Link in a materialized view when the dimension can arrive late.

**Pitfall: watermarks drop silently.** Anything stateful with a watermark discards data that arrives too late. Always pair it with a count that would show the loss.

**Pitfall: SCD2 on bookkeeping columns.** If `_ingest_ts` counted as a change, every re-read would add a new "version" with identical content.

**Next:** notebook 13 builds Gold: patient-hour features and labels that match `features.py`, and the point-in-time rules that keep the model honest.